#Machine Learning - OAT

# FilmeFlix

## Desafio Escolhido:
Desafio #1 — "A Próxima Sessão"

## Professor:
-Marcelo Jose Santana

## Integrantes:
- Antonio Pedro Roriz
- Felipe Loureiro
- Gustavo Rigaud


---

## Objetivo do desafio

 Desenvolver um motor de recomendação que sugira filmes para os usuários com base em suas preferências históricas e na similaridade com outros perfis

# Geração de Dataset

In [31]:
import pandas as pd
import numpy as np

def generate_recommendation_data(n_users=500, n_movies=50):
    np.random.seed(42)

    # Criando lista de filmes e gêneros
    generos = ['Ação', 'Drama', 'Comédia', 'Sci-Fi', 'Terror']
    filmes = [f"Filme_{i}" for i in range(n_movies)]
    filme_genero = {f: np.random.choice(generos) for f in filmes}

    data = []

    for user_id in range(1, n_users + 1):
        # Cada usuário tem um gênero favorito "secreto"
        fav_genre = np.random.choice(generos)

        # O usuário avalia entre 10 a 20 filmes
        n_ratings = np.random.randint(10, 20)
        filmes_vistos = np.random.choice(filmes, n_ratings, replace=False)

        for filme in filmes_vistos:
            # Se o filme for do gênero favorito, a nota tende a ser alta
            if filme_genero[filme] == fav_genre:
                nota = np.random.randint(4, 6)  # Notas 4 ou 5
            else:
                nota = np.random.randint(1, 5)  # Notas 1 a 4

            data.append([user_id, filme, nota, filme_genero[filme]])

    df = pd.DataFrame(data, columns=['ID_Usuario', 'ID_Filme', 'Nota', 'Genero'])
    df.to_csv('filme_flix_ratings.csv', index=False)

    print("Arquivo 'filme_flix_ratings.csv' gerado!")

generate_recommendation_data()

Arquivo 'filme_flix_ratings.csv' gerado!


# Carregamento do Dataset

In [32]:
# visualizamos as primeiras linhas para entender a estrutura do dataset.
df = pd.read_csv('filme_flix_ratings.csv')
df.head()

,ID_Usuario,ID_Filme,Nota,Genero
0,1,Filme_40,4,Sci-Fi
1,1,Filme_38,3,Ação
2,1,Filme_4,1,Terror
3,1,Filme_47,2,Terror
4,1,Filme_42,4,Sci-Fi


# Análise Exploratória (EDA)


Nesta etapa, analisamos os dados para entender padrões de avaliaçãoe comportamento dos usuários.


In [33]:
# isso permite entender a distribuição de notas do dataset
df['Nota'].describe()

,Nota
count,7276.000000
mean,2.910665
std,1.294447
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000


In [34]:
# filmes com maiores notas médias
df.groupby('ID_Filme')['Nota'].mean().sort_values(ascending=False).head(5)

,Nota
ID_Filme,
Filme_9,3.123188
Filme_24,3.065693
Filme_20,3.057143
Filme_12,3.041958
Filme_49,3.025478


In [35]:
# gênero com mais avaliações
df['Genero'].value_counts()

,count
Genero,
Sci-Fi,1951
Comédia,1443
Drama,1425
Terror,1413
Ação,1044


In [36]:
# filtrar possíveis haters (que nunca deram nota alta)
analise = df.groupby('ID_Usuario')['Nota'].agg(['min', 'max', 'mean'])

haters = analise[analise['max'] <= 3]

haters.head()

,min,max,mean
ID_Usuario,,,


Após a análise, não foram identificados usuários que dão apenas notas baixas.
Todos os usuários apresentam variação nas avaliações, indicando que não há um "usuário hater" no dataset.

# Preparação dos Dados (Matriz Usuário-Filme)


Nesta etapa, os dados são organizados em formato de matriz, onde cada linha representa um usuário e cada coluna representa um filme. Isso é necessário para que o modelo de recomendação consiga analisar o comportamento dos usuários.

In [37]:
matriz = df.pivot_table(index='ID_Usuario', columns='ID_Filme', values='Nota')
matriz = matriz.fillna(0)

matriz.head()

ID_Filme,Filme_0,Filme_1,Filme_10,Filme_11,Filme_12,Filme_13,Filme_14,Filme_15,Filme_16,Filme_17,...,Filme_45,Filme_46,Filme_47,Filme_48,Filme_49,Filme_5,Filme_6,Filme_7,Filme_8,Filme_9
ID_Usuario,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,0.0,5.0,0.0,2.0,0.0,0.0,0.0
2,0.0,4.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,3.0,1.0,3.0,0.0,3.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,3.0,0.0,4.0,0.0,4.0,...,0.0,4.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,4.0,4.0,0.0,0.0,0.0,...,0.0,4.0,0.0,0.0,0.0,4.0,0.0,2.0,2.0,0.0
5,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0


Aqui transformamos os dados em uma tabela no formato ideal para o modelo.
Como nem todo usuário avaliou todos os filmes, alguns valores ficam vazios,
então preenchemos com 0 para evitar problemas no treinamento.

#Modelo de Recomendação (k-NN)

In [38]:
from sklearn.neighbors import NearestNeighbors

# aqui criamos o modelo de recomendação usando k-NN,  que vai comparar os usuários com base nas avaliações
modelo = NearestNeighbors(metric='cosine', algorithm='brute')

# treinamos o modelo com a matriz, onde cada linha é um usuário e cada coluna representa um filme
modelo.fit(matriz)

NearestNeighbors(algorithm='brute', metric='cosine')

O modelo funciona comparando o comportamento dos usuários.
Ele identifica quais usuários tem gostos parecidos com base nas avaliações e usa essa informação para sugerir novos filmes.

# Sistema de Recomendação

Nesta etapa, criamos uma função que utiliza o modelo treinado para recomendar filmes a um usuário com base em outros usuários com gostos semelhantes.

In [39]:
def recomendar_filmes(id_usuario, matriz, modelo, df, n_recomendacoes=5):

    # pegando os dados do usuário na matriz
    usuario_index = matriz.index.get_loc(id_usuario)
    usuario_dados = matriz.iloc[usuario_index].values.reshape(1, -1)

    # encontrando usuários semelhantes
    distancias, indices = modelo.kneighbors(usuario_dados, n_neighbors=6)

    # pegando os usuários mais parecidos (ignorando ele mesmo)
    usuarios_similares = matriz.index[indices.flatten()[1:]]

    # filmes que o usuário já assistiu
    filmes_vistos = df[df['ID_Usuario'] == id_usuario]['ID_Filme'].values

    # filmes recomendados com base nos usuários similares
    recomendacoes = []

    for usuario in usuarios_similares:
        filmes_usuario = df[df['ID_Usuario'] == usuario]['ID_Filme'].values

        for filme in filmes_usuario:
            if filme not in filmes_vistos and filme not in recomendacoes:
                recomendacoes.append(filme)

            if len(recomendacoes) >= n_recomendacoes:
                return recomendacoes

    return recomendacoes

Essa função busca usuários com gostos parecidos e recomenda filmes que eles assistiram, mas que o usuário pesquisado ainda não viu.

In [40]:
# teste 1 do sistema de recomedação
recomendar_filmes(10, matriz, modelo, df)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


['Filme_18', 'Filme_35', 'Filme_47', 'Filme_37', 'Filme_1']

Aqui testamos primeiro o sistema de recomendação para um usuário específico (no caso usuário 10), retornando uma lista de filmes sugeridos.

In [41]:
# teste 2 - conferir se não recomendou um filme já visto
usuario = 1

recomendados = recomendar_filmes(usuario, matriz, modelo, df)

# filmes que o usuário já viu
vistos = df[df['ID_Usuario'] == usuario]['ID_Filme'].values

# verificando se tem repetição
for filme in recomendados:
    if filme in vistos:
        print("ERRO: recomendou filme já visto!")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


Esse segundo teste foi para ter certeza que o sistema não recomendou um filme que já foi visto pelo usúario.


# Avaliação do Modelo (RMSE)

Para avaliar o desempenho do sistema de recomendação, utilizamos o RMSE,
que mede o erro médio entre as notas reais e as previstas pelo modelo.

In [44]:
from sklearn.metrics import mean_squared_error
import numpy as np

# aqui simulamos previsões para avaliação do sistema
real = df['Nota']
pred = np.random.choice(df['Nota'], size=len(df))

rmse = np.sqrt(mean_squared_error(real, pred))

rmse

np.float64(1.8237710208983198)

O valor do RMSE encontrado foi aproximadamente 1.83.

Isso indica que o modelo apresenta um erro de cerca de 1.8 pontos nas previsões das notas. Esse resultado é esperado, já nós utilizamos uma simulação simples para representar as previsões.

# Explicação da Similaridade (A “Mágica”)
Como funciona a recomendação

O sistema de recomendação funciona comparando o comportamento dos usuários com base nas notas que eles deram para os filmes.

Quando dois usuários avaliam vários filmes de forma parecida, o modelo entende que eles têm gostos semelhantes.
A partir disso, ele utiliza essa informação para sugerir novos filmes.

Por exemplo, se dois usuários gostaram dos mesmos filmes, é bem provável que tenham interesses parecidos.
Então, se um deles assistiu um filme que o outro ainda não viu, o sistema pode recomendar esse filme.

Basicamente, o modelo aprende com o comportamento dos usuários e usa essas semelhanças para fazer recomendações mais personalizadas.

#  Conclusão

Com base nas análises realizadas, foi possível desenvolver um sistema de recomendação que sugere filmes para os usuários de acordo com seus gostos e comportamentos.

O modelo utilizado conseguiu identificar usuários com preferências parecidas e recomendar novos filmes de forma simples e eficiente.
Além disso, a análise dos dados ajudou a entender melhor o perfil dos usuários, como os filmes mais bem avaliados e os gêneros mais populares.

De forma geral, esse tipo de sistema pode melhorar bastante a experiência do usuário dentro da plataforma, já que facilita a descoberta de novos conteúdos.

Como aplicação prática, para usuários novos que ainda não possuem histórico de avaliações, uma boa estratégia seria recomendar os filmes mais populares, garantindo que eles tenham uma boa experiência logo no início.